# Generate Skill — Notebook Control

ManiSkill 데모를 리플레이해서 (image + action) HDF5 데이터셋을 만드는 스킬의 호출 검증.

두 가지 호출 방식을 모두 검증:
1. **Function mode** — `from scripts.generate import run` (노트북 패널이 주로 쓸 패턴)
2. **Subprocess mode** — 별도 프로세스로 CLI 호출 (격리 실행 / 외부 도구 패턴)


## Setup — scripts 모듈 import 경로 설정

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('project root:', PROJECT_ROOT)
print('scripts/ exists:', (PROJECT_ROOT / 'scripts').is_dir())

project root: C:\Users\hun41\PycharmProjects\maniskill
scripts/ exists: True


## ① Function mode — 모듈 import 호출

In [2]:
from scripts.generate import run

out_func = run(task='PickCube-v1', count=3, obs_mode='rgb')
print('returned path:', out_func)
print('exists:', out_func.exists(), '| size: {:.1f} KB'.format(out_func.stat().st_size / 1024))

C:\Users\hun41\miniconda3\envs\maniskill\Lib\site-packages\sapien\wrapper\pinocchio_model.py:299: UserWarning: pinnochio package is not installed, robotics functionalities will not be available
  warnings.warn(


  0%|          | 0/3 [00:00<?, ?step/s]

0step [00:00, ?step/s]

2026-06-01 16:41:44,015 - mani_skill  - WARNING - shader_dir argument will be deprecated after ManiSkill v3.0.0 official release. Please use sensor_configs/human_render_camera_configs to set shaders.


2026-06-01 16:41:44,975 - mani_skill  - WARNING - mani_skill is not installed with git.


...

Replaying traj_2:  50%|█████     | 25/50 [00:00<00:00, 238.69step/s]

Replaying traj_2:  98%|█████████▊| 49/50 [00:00<00:00, 224.85step/s]

Replaying traj_2: 100%|██████████| 50/50 [00:00<00:00, 133.17step/s]


  0%|          | 0/3 [00:02<?, ?step/s]

Replayed 3 episodes, 3/3=100.00% demos saved
returned path: C:\Users\hun41\.maniskill\demos\PickCube-v1\motionplanning\trajectory.rgb.pd_joint_pos.physx_cpu.h5
exists: True | size: 1144.7 KB


## ② Subprocess mode — 별도 프로세스 CLI 호출

In [3]:
import subprocess, sys as _sys

result = subprocess.run(
    [_sys.executable, str(PROJECT_ROOT / 'scripts' / 'generate.py'),
     '--task', 'PickCube-v1', '--count', '3', '--obs-mode', 'rgb'],
    capture_output=True, text=True,
)
print('returncode:', result.returncode)
print('--- last 8 stdout lines ---')
print('\n'.join(result.stdout.strip().splitlines()[-8:]))

returncode: 0
--- last 8 stdout lines ---
Replayed 3 episodes, 3/3=100.00% demos saved

Generated dataset -> C:\Users\hun41\.maniskill\demos\PickCube-v1\motionplanning\trajectory.rgb.pd_joint_pos.physx_cpu.h5


## ③ 결과 HDF5 간단 검증 — 두 방식이 같은 산출물 만들었는지

In [4]:
import h5py

with h5py.File(out_func, 'r') as f:
    trajs = sorted(f.keys())
    sample = f[trajs[0]]
    rgb_shape = sample['obs']['sensor_data']['base_camera']['rgb'].shape
    act_shape = sample['actions'].shape
print('episodes in file:', len(trajs))
print('episode 0 RGB shape:', rgb_shape)
print('episode 0 actions shape:', act_shape)

episodes in file: 3
episode 0 RGB shape: (75, 128, 128, 3)
episode 0 actions shape: (74, 8)
